In [ ]:
# Import path to source directory (bit of a hack in Jupyter)
import sys
import os
pwd = %pwd
sys.path.append(os.path.join(pwd, '..'))
sys.path.append(os.path.join(pwd, '.'))

# Ensure modules are reloaded on any change (very useful when developing code on the fly)
%load_ext autoreload
%autoreload 2

In [ ]:
%matplotlib inline
from programmable_cubes_UDP import ProgrammableCubes
from programmable_cubes_UDP import programmable_cubes_UDP
import numpy as np
from pygmo import problem
import random
from src.implementation_heuristic import *
from src.visual import *
from tqdm import tqdm

In [ ]:
PROBLEM = "ISS_INV"
udp = programmable_cubes_UDP(PROBLEM,"../")
udp.fitness(np.array([-1]))
ti = udp.initial_cube_types
ci = udp.final_cube_positions
ct = udp.target_cube_positions
tt = udp.target_cube_types
types = np.arange(np.max(ti)+1)

udp.plot("ensemble",types)
udp.plot("target",types)



## Experimental code for removing stuck cubes using bridges

In [ ]:
final_config = np.array(udp.final_cube_positions)

wrong_ids_active, wrong_ids_target = get_wrong_ids_and_coords(udp)

cubes = ProgrammableCubes(final_config)
stuck = get_stuck_wrong_cubes(cubes,wrong_ids_active)
freeroaming = get_freeroaming_cubes(cubes,np.arange(len(final_config)))


stuck_id = stuck[0]
stuck_position = cubes.cube_position[stuck_id]
stuck = True

neigbour_positions = cubes.cube_position[cubes.cube_neighbours[stuck_id]]
relative_neigbour_positions = neigbour_positions-stuck_position
print(relative_neigbour_positions)
## Find out which direction to make the bridge in
# figuring out L shapes now
# direction in which they dont change at all
common_zero = np.sum(relative_neigbour_positions == np.array([0,0,0]),axis=0) == len(relative_neigbour_positions)
direction_of_bridge = np.where(common_zero)[0][0]
basis = np.array([[1,0,0],[0,1,0],[0,0,1]])
relative_vectors = np.zeros(shape=(5,3))
relative_vectors[0] = basis[direction_of_bridge]
relative_vectors[1] = basis[direction_of_bridge] + basis[(direction_of_bridge+1)%3]
relative_vectors[2] = basis[direction_of_bridge] - basis[(direction_of_bridge+1)%3]
relative_vectors[3] = basis[direction_of_bridge] + basis[(direction_of_bridge+2)%3]
relative_vectors[4] = basis[direction_of_bridge] - basis[(direction_of_bridge+2)%3]
print(relative_vectors)
bridge_coords = relative_vectors + stuck_position
print(stuck_position)
print(bridge_coords)



stuck_id + DIRS[direction_of_bridge]
print(direction_of_bridge)



for i in range(len(bridge_coords)):
    tmp_chrom, tmp_path, success = astar_cubes(cubes,freeroaming[i],bridge_coords[i])
    if success:
        cubes.apply_chromosome(end_chromosome(tmp_chrom),True)
        chrom = np.concatenate([chrom,tmp_chrom])
udp.plot("none",types,cubes.cube_position,ti)
udp.fitness(end_chromosome(chrom))
print(astar_cubes(cubes,stuck_id,np.array([-1,0,0])))
udp.plot("none",types,cubes.cube_position,ti)
print(force_random_move(stuck_id,cubes))



In [ ]:
chrom = apply_find_chromosome_multiple_times(udp,find_chromosome_heuristic,2,chrom)
udp.fitness(end_chromosome(chrom))
udp.plot("ensemble",types)

## Make it into one method

In [ ]:



chrom = np.array([],dtype=int)

print("Running first heuristics")
udp.fitness(end_chromosome(chrom))
chrom = apply_find_chromosome_multiple_times(udp,find_chromosome_heuristic,5,chrom)
udp.fitness(end_chromosome(chrom))

print("Running removal of stuck cubes")
chrom = run_removal_of_stuck_cubes_on_udp(udp,chrom,10)
#chrom = chrom
print(len(chrom),udp.fitness(end_chromosome(chrom)))

# print("Running post heuristics")
# chrom = apply_find_chromosome_multiple_times(udp,find_chromosome_heuristic,5,chrom)
# print(len(chrom),udp.fitness(end_chromosome(chrom)))

# print("Running removal of stuck cubes")
# chrom = run_removal_of_stuck_cubes_on_udp(udp,chrom,50)
# print(len(chrom),udp.fitness(end_chromosome(chrom)))

# print("Running post heuristics")
# chrom = apply_find_chromosome_multiple_times(udp,find_chromosome_heuristic,5,chrom)
# print(len(chrom),udp.fitness(end_chromosome(chrom)))

# print("Running removal of stuck cubes")
# chrom = run_removal_of_stuck_cubes_on_udp(udp,chrom,50)
# print(len(chrom),udp.fitness(end_chromosome(chrom)))

# print("Running post heuristics")
# chrom = apply_find_chromosome_multiple_times(udp,find_chromosome_heuristic,5,chrom)
# print(len(chrom),udp.fitness(end_chromosome(chrom)))

# chrom = apply_find_chromosome_multiple_times(udp,find_chromosome,5,chrom)
# print(len(chrom),udp.fitness(end_chromosome(chrom)))


# chrom = apply_find_chromosome_multiple_times(udp,find_chromosome_heuristic,5,chrom)
# print(len(chrom),udp.fitness(end_chromosome(chrom)))

udp.fitness(end_chromosome(chrom))
print(analyze_first_and_second_mistakes(udp))

In [ ]:
print("Running removal of stuck cubes")
chrom = run_removal_of_stuck_cubes_on_udp(udp,chrom,10,True)
print(len(chrom),udp.fitness(end_chromosome(chrom)))

In [ ]:
chrom = apply_find_chromosome_multiple_times(udp,find_chromosome_heuristic,5,chrom)
print(len(chrom),udp.fitness(end_chromosome(chrom)))


In [ ]:
print(analyze_first_and_second_mistakes(udp))
np.save("out",chrom)
udp.plot("ensemble",types)
udp.plot("target",types)

In [ ]:
save_achieved_config(udp,chrom)
save_mistakes(udp)